In [ ]:
import os

import httpx
import openai
from dotenv import find_dotenv, load_dotenv


load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 설정되어 있지 않습니다."

client = openai.OpenAI()
MODEL = "gpt-4o-mini"
BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

In [ ]:
# --- 에이전트가 실제로 호출할 수 있는 함수들 ---


def get_popular_movies():
    """/movies 에서 인기 영화 목록을 가져옵니다."""
    return httpx.get(f"{BASE_URL}/movies", timeout=20).json()


def get_movie_details(id):
    """/movies/:id 에서 영화 상세 정보를 가져옵니다."""
    return httpx.get(f"{BASE_URL}/movies/{id}", timeout=20).json()


def get_movie_credits(id):
    """/movies/:id/credits 에서 출연진·제작진을 가져옵니다."""
    return httpx.get(f"{BASE_URL}/movies/{id}/credits", timeout=20).json()


FUNCTIONS = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

In [ ]:

PROMPT = """
I have the following functions in my movie system.

'get_popular_movies'   -> takes NO argument.        Returns currently popular movies.
'get_movie_details'    -> takes a movie id (number). Returns details of that movie.
'get_movie_credits'    -> takes a movie id (number). Returns the cast and crew of that movie.

Choose the single function that best answers the user's question.

Please answer ONLY with the function name and its arguments, nothing else.
Examples of valid answers:
get_popular_movies()
get_movie_details(550)
get_movie_credits(550)

Answer the following question:

{question}
"""

In [ ]:
def decide_function(question):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": PROMPT.format(question=question)},
        ],
    )
    return response.choices[0].message.content.strip()


for q in [
    "지금 인기 있는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘",
]:
    print(f"Q: {q}")
    print(f"-> {decide_function(q)}\n")

In [1]:
import re


def run_agent(question):
    """질문 -> 모델이 함수 선택 -> 실제 함수 호출까지 한 번에 수행합니다."""
    call = decide_function(question)
    print(f"Q: {question}")
    print(f"모델이 고른 함수: {call}")

    match = re.match(r"\s*(\w+)\s*\((.*?)\)\s*", call)
    if not match:
        raise ValueError(f"함수 호출을 파싱할 수 없습니다: {call!r}")

    name, raw_args = match.group(1), match.group(2).strip()
    func = FUNCTIONS[name]
    args = [int(raw_args)] if raw_args else []  

    return func(*args)


result = run_agent("movie ID 550에 해당하는 영화가 무엇인지 알려줘")
print("제목:", result["title"])
print("개요:", result["overview"][:120], "...")

NameError: name 'decide_function' is not defined

## 🎬 취향을 기억하는 영화 추천 챗봇

대화 기록(`messages`)을 메모리로 활용해 사용자의 취향을 기억하는 챗봇입니다.

- 좋아하는 **장르**를 기억
- 이미 **시청한 영화**를 기억
- 대화 기록 기반 **개인화 추천** (이미 본 영화는 추천 제외)

**핵심 아이디어:** 매 턴마다 *지금까지의 전체 대화 기록*을 모델에 함께 전달하면, 그 대화 기록 자체가 메모리가 됩니다. 사용자 메시지(`role: "user"`)와 AI 응답(`role: "assistant"`)을 모두 `messages` 리스트에 `append` 하는 것이 핵심입니다.

In [ ]:
# --- 취향을 기억하는 영화 추천 챗봇 ---

SYSTEM_PROMPT = """
당신은 사용자의 영화 취향을 기억하는 친절한 영화 추천 챗봇입니다.

규칙:
1. 사용자가 좋아한다고 말한 장르를 기억하세요.
2. 사용자가 이미 봤다고 말한 영화를 기억하세요.
3. 추천할 때는 사용자가 좋아하는 장르를 우선하되, 이미 본 영화는 절대 추천하지 마세요.
4. 한국어로 자연스럽고 친근하게 대화하세요.
""".strip()

# 대화 기록을 저장할 messages 리스트.
# system 메시지로 시작하고, 이후 user/assistant 메시지가 계속 누적됩니다.
# 이 리스트 전체를 매 턴 모델에 전달하는 것이 곧 "메모리"입니다.
messages = [{"role": "system", "content": SYSTEM_PROMPT}]


def chat(user_input):
    # 1) 사용자 메시지를 대화 기록에 추가
    messages.append({"role": "user", "content": user_input})

    # 2) 지금까지의 전체 대화 기록을 전달 -> 모델이 이전 대화를 모두 "기억"
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
    )
    answer = response.choices[0].message.content.strip()

    # 3) AI 응답도 대화 기록에 추가 (다음 턴에서 기억됨)
    messages.append({"role": "assistant", "content": answer})

    print(f"User: {user_input}")
    print(f"AI: {answer}\n")
    return answer

In [ ]:
# --- 챗봇 테스트: 4턴 대화 ---
# 같은 messages 리스트를 공유하므로, 앞선 대화를 모두 기억합니다.

chat("나는 SF 영화를 좋아해")
chat("인셉션이랑 인터스텔라는 이미 봤어")
chat("오늘 밤에 뭐 볼지 추천해 줄래?")
chat("내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?")

In [ ]:
# --- 대화 기록(메모리) 전체 확인 ---
# user 메시지와 assistant 응답이 모두 누적되어 있습니다.
# 이 리스트 자체가 챗봇의 "기억"입니다.

print(f"총 {len(messages)}개의 메시지가 저장되어 있습니다.\n")
for m in messages:
    content = m["content"].replace("\n", " ")
    print(f"[{m['role']:<9}] {content[:70]}")